# HyDE + Hybrid RAG
Choose insurance or Python documentation. Install requirements using the README first. This notebook has not been executed; saved benchmark files are separate. InsuranceQA is a historical research corpus, not current policy advice.

In [ ]:
from pathlib import Path
import os
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
from src import state
from src import pipeline as rag
state.select_dataset("insurance")
question = "Is workers compensation the same as disability insurance?"

## 1. Prepare and index documents

In [ ]:
from prepare_data import prepare
prepare(state.DATASET)
chunks = rag.initialize(128)
print(len(chunks), "chunks")
chunks[0]

## 2. Generate the hypothetical document

In [ ]:
info = rag.query_embedding(question, use_hyde=True)
print(info["hypothetical"])

## 3. Search both indexes
BM25 uses the original question. Dense search uses the hypothetical document embedding.

In [ ]:
lexical = rag.bm25_search(question, 30)
dense = rag.dense_search(info["vector"], 30)
print("BM25", [(r["chunk_id"], r["score"]) for r in lexical[:5]])
print("Dense", [(r["chunk_id"], r["score"]) for r in dense[:5]])

## 4. Normalize and fuse
Alpha is the BM25 weight. Missing candidates contribute zero. Chunk IDs remove duplicates.

In [ ]:
fused = rag.fuse_results(lexical, dense, alpha=0.5)
[(r["chunk_id"], r["score"]) for r in fused[:5]]

## 5. Rerank and generate from actual source text

In [ ]:
ranked = rag.rerank(question, fused[:30], top_k=10)
answer = rag.generate_answer(question, ranked, rag.GENERATOR)
print(answer["answer"])
print(answer["sources"])
answer["context"]

## 6. Inspect saved results

In [ ]:
import csv
with open("results/experiment_summary.csv", encoding="utf-8") as file:
    results = list(csv.DictReader(file))
results

## 7. Full evaluation (optional, can take a long time on CPU)
Runs E1–E6 at both sizes and three fusion weights; selects weights using development questions. Writes to results/<dataset>/. Enable the flag to run.

In [ ]:
RUN_FULL_EVALUATION = False
if RUN_FULL_EVALUATION:
    from src.evaluation import run_experiments
    from src.results import refresh_results
    run_experiments()
    refresh_results()